# Exogenous Data Sources — Ingestion

Fetches macro (NDX, DXY), Fear & Greed, Binance funding rate, Google Trends,
and Reddit sentiment data, then merges everything onto the BTC date index.

**Run this notebook before re-running `02_feature_analysis.ipynb`.**

Missing-history policy: pre-inception gaps stay `NaN` with a `<feature>_missing`
flag (never imputed with a neutral value). Calendar/resolution gaps
(weekends for NDX/DXY, weekly resolution for Google Trends) are forward-filled,
each with their own `_missing` flag marking interpolated days.

In [1]:
import yfinance as yf

# Nasdaq Composite = ^IXIC (NOT ^NDX, which is the Nasdaq-100)
try:
    macro_tickers = ["^IXIC", "DX-Y.NYB"]
    macro_data = yf.download(macro_tickers, period="10y", interval="1d")
    macro_data.to_csv("../data/raw/yahoo_macro.csv")
    print(macro_data.shape)
    print(macro_data.tail())
except Exception as e:
    print(f"WARNING: yahoo_macro fetch failed ({e}) - downstream merge will treat this source as unavailable")

/var/folders/dy/8mkjsrmx33v0zw8njf1n2bxr0000gn/T/ipykernel_65534/487012001.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  macro_data = yf.download(macro_tickers, period="10y", interval="1d")


[                       0%                       ]

[*********************100%***********************]  2 of 2 completed

(2515, 10)
Price           Close                      High                      Low  \
Ticker       DX-Y.NYB         ^IXIC    DX-Y.NYB         ^IXIC   DX-Y.NYB   
Date                                                                       
2026-08-06  99.970001  26348.349609  100.019997  26499.419922  99.639999   
2026-08-07  99.599998  26690.619141  100.000000  26712.619141  99.400002   
2026-08-10  99.809998  26605.359375   99.830002  26724.630859  99.580002   
2026-08-11  99.820000  26445.449219   99.900002  26679.259766  99.730003   
2026-08-12  99.747002  26581.183594   99.902000  26688.240234  99.612999   

Price                          Open                 Volume                
Ticker             ^IXIC   DX-Y.NYB         ^IXIC DX-Y.NYB         ^IXIC  
Date                                                                      
2026-08-06  26208.429688  99.660004  26268.839844      0.0  8.936850e+09  
2026-08-07  26478.009766  99.940002  26534.660156      0.0  8.183970e+09  
2026-

In [2]:
import requests
import pandas as pd

try:
    resp = requests.get("https://api.alternative.me/fng/?limit=0&format=json", timeout=30)
    resp.raise_for_status()
    fng_raw = resp.json()["data"]

    fng = pd.DataFrame(fng_raw)
    fng["Date"] = pd.to_datetime(fng["timestamp"].astype(int), unit="s").dt.normalize()
    fng["fear_greed_value"] = fng["value"].astype(float)
    fng = fng.rename(columns={"value_classification": "fear_greed_classification"})
    fng = fng[["Date", "fear_greed_value", "fear_greed_classification"]].sort_values("Date")
    fng = fng.set_index("Date")
    fng.to_csv("../data/raw/fear_greed.csv")
    print(fng.shape)
    print(fng.head())
    print(fng.tail())
except Exception as e:
    print(f"WARNING: fear_greed fetch failed ({e}) - downstream merge will treat this source as unavailable")

(3111, 2)
            fear_greed_value fear_greed_classification
Date                                                  
2018-02-01              30.0                      Fear
2018-02-02              15.0              Extreme Fear
2018-02-03              40.0                      Fear
2018-02-04              24.0              Extreme Fear
2018-02-05              11.0              Extreme Fear
            fear_greed_value fear_greed_classification
Date                                                  
2026-08-08              30.0                      Fear
2026-08-09              31.0                      Fear
2026-08-10              30.0                      Fear
2026-08-11              29.0                      Fear
2026-08-12              27.0                      Fear


In [3]:
import requests
import pandas as pd
import time

def fetch_binance_funding_rates(symbol="BTCUSDT", start_time_ms=1567900800000):
    """start_time_ms defaults to 2019-09-08, before BTCUSDT perpetual launch (2019-09-13)."""
    url = "https://fapi.binance.com/fapi/v1/fundingRate"
    rows = []
    start = start_time_ms
    while True:
        resp = requests.get(url, params={"symbol": symbol, "startTime": start, "limit": 1000}, timeout=30)
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break
        rows.extend(batch)
        last_time = batch[-1]["fundingTime"]
        if last_time <= start:
            break
        start = last_time + 1
        if len(batch) < 1000:
            break
        time.sleep(0.3)
    return rows

try:
    funding_rows = fetch_binance_funding_rates()
    funding = pd.DataFrame(funding_rows)
    funding["Date"] = pd.to_datetime(funding["fundingTime"], unit="ms").dt.normalize()
    funding["fundingRate"] = funding["fundingRate"].astype(float)
    funding_daily = funding.groupby("Date")["fundingRate"].mean().rename("funding_rate").to_frame()
    funding_daily.to_csv("../data/raw/funding_rates.csv")
    print(funding_daily.shape)
    print(funding_daily.head())
    print(funding_daily.tail())
except Exception as e:
    print(f"WARNING: funding_rates fetch failed ({e}) - downstream merge will treat this source as unavailable")

(2529, 1)
            funding_rate
Date                    
2019-09-10        0.0001
2019-09-11        0.0001
2019-09-12        0.0001
2019-09-13        0.0001
2019-09-14        0.0001
            funding_rate
Date                    
2026-08-08      0.000056
2026-08-09      0.000060
2026-08-10      0.000068
2026-08-11      0.000042
2026-08-12      0.000093


In [4]:
from pytrends.request import TrendReq
import pandas as pd
import time

def fetch_google_trends(keyword="Bitcoin", timeframe="2016-01-01 2026-08-12", retries=3):
    pytrends = TrendReq(hl="en-US", tz=0)
    for attempt in range(retries):
        try:
            pytrends.build_payload([keyword], timeframe=timeframe)
            return pytrends.interest_over_time()
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f"Google Trends request failed ({e}), retrying in 10s...")
            time.sleep(10)

try:
    trends = fetch_google_trends()
    trends = trends.drop(columns=["isPartial"], errors="ignore")
    trends = trends.rename(columns={"Bitcoin": "google_trends_score"})
    trends.index.name = "Date"
    trends.to_csv("../data/raw/google_trends.csv")
    print(trends.shape)
    print(trends.head())
    print(trends.tail())
except Exception as e:
    print(f"WARNING: google_trends fetch failed after retries ({e}) - downstream merge will treat this source as unavailable")

(128, 1)
            google_trends_score
Date                           
2016-01-01                    3
2016-02-01                    3
2016-03-01                    3
2016-04-01                    3
2016-05-01                    4
            google_trends_score
Date                           
2026-04-01                   25
2026-05-01                   23
2026-06-01                   27
2026-07-01                   21
2026-08-01                   15


/Users/vivienbistrel/AI4Finance/.venv/lib/python3.13/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


In [5]:
import os
import pandas as pd

btc = pd.read_csv("../data/processed/yahooFinanceDataCleaned.csv", index_col=0)
btc.index = pd.to_datetime(btc.index)
date_index = btc.index

exogenous = pd.DataFrame(index=date_index)
exogenous.index.name = "Date"


def load_raw_csv(path, header=0):
    if not os.path.exists(path):
        print(f"WARNING: {path} not found, its columns will be entirely NaN/missing")
        return None
    df = pd.read_csv(path, index_col=0, header=header)
    df.index = pd.to_datetime(df.index)
    return df


# --- Macro (NDX, DXY): market-closed gaps get forward-filled ---
macro_raw = load_raw_csv("../data/raw/yahoo_macro.csv", header=[0, 1])
if macro_raw is not None:
    macro = pd.DataFrame(index=macro_raw.index)
    macro["NDX_Close"] = macro_raw["Close"]["^IXIC"]
    macro["DXY_Close"] = macro_raw["Close"]["DX-Y.NYB"]
    macro = macro.reindex(date_index)
else:
    macro = pd.DataFrame({"NDX_Close": pd.NA, "DXY_Close": pd.NA}, index=date_index)

for col in ["NDX_Close", "DXY_Close"]:
    exogenous[f"{col}_missing"] = macro[col].isna().astype(int)
    exogenous[col] = macro[col].ffill()

# --- Fear & Greed: genuine pre-2018 non-existence, NaN stays NaN ---
fng = load_raw_csv("../data/raw/fear_greed.csv")
fng = fng.reindex(date_index) if fng is not None else pd.DataFrame({"fear_greed_value": pd.NA}, index=date_index)
exogenous["fear_greed_value"] = fng["fear_greed_value"]
exogenous["fear_greed_value_missing"] = fng["fear_greed_value"].isna().astype(int)

# --- Funding rate: genuine pre-2019-09 non-existence, NaN stays NaN ---
funding = load_raw_csv("../data/raw/funding_rates.csv")
funding = funding.reindex(date_index) if funding is not None else pd.DataFrame({"funding_rate": pd.NA}, index=date_index)
exogenous["funding_rate"] = funding["funding_rate"]
exogenous["funding_rate_missing"] = funding["funding_rate"].isna().astype(int)

# --- Google Trends: native since 2016 but weekly resolution, forward-filled ---
trends = load_raw_csv("../data/raw/google_trends.csv")
trends = trends.reindex(date_index) if trends is not None else pd.DataFrame({"google_trends_score": pd.NA}, index=date_index)
exogenous["google_trends_score_missing"] = trends["google_trends_score"].isna().astype(int)
exogenous["google_trends_score"] = trends["google_trends_score"].ffill()

# --- Reddit: only its fetch window has data, NaN stays NaN outside it ---
reddit_daily = load_raw_csv("../data/raw/reddit_sentiment.csv")
if reddit_daily is not None:
    reddit_daily = reddit_daily.reindex(date_index)
else:
    reddit_daily = pd.DataFrame({"reddit_polarity": pd.NA, "reddit_volume": pd.NA}, index=date_index)
for col in ["reddit_polarity", "reddit_volume"]:
    exogenous[f"{col}_missing"] = reddit_daily[col].isna().astype(int)
    exogenous[col] = reddit_daily[col]

exogenous.to_csv("../data/raw/exogenous_merged.csv")
print(exogenous.shape)
print(exogenous.isna().mean().rename("pct_nan"))


(3653, 14)
NDX_Close_missing              0.000000
NDX_Close                      0.020257
DXY_Close_missing              0.000000
DXY_Close                      0.020257
fear_greed_value               0.168629
fear_greed_value_missing       0.000000
funding_rate                   0.327950
funding_rate_missing           0.000000
google_trends_score_missing    0.000000
google_trends_score            0.000547
reddit_polarity_missing        0.000000
reddit_polarity                1.000000
reddit_volume_missing          0.000000
reddit_volume                  1.000000
Name: pct_nan, dtype: float64
